In [14]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

train_df = pd.read_csv(r'D:\Archmooner\python\ML\Datasets\titanic\train.csv')
test_df = pd.read_csv(r'D:\Archmooner\python\ML\Datasets\titanic\test.csv')

combine = [train_df, test_df]

In [15]:
for df in combine:
    # title: extract title from name
    df["Title"] = df["Name"].str.extract(r" ([A-Za-z]+)\.", expand=False)    # group rare titles into one category
    df['Title'] = df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')

    df['Title'] = df['Title'].replace('Mlle', 'Miss')
    df['Title'] = df['Title'].replace('Ms', 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')

    # feature family size & isAlone
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = 0
    df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

    # impute missing values
    df["Age"] = df["Age"].fillna(train_df["Age"].median())
    df["Fare"] = df["Fare"].fillna(train_df["Fare"].median())
    df["Embarked"] = df["Embarked"].fillna(train_df["Embarked"].mode()[0])

    # encode categorical variables
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

In [16]:
# One-hot encoding untuk Embarked dan Title
train_df = pd.get_dummies(train_df, columns=["Embarked", "Title"], drop_first=True)
test_df = pd.get_dummies(test_df, columns=["Embarked", "Title"], drop_first=True)

# Samakan kolom test_df agar persis dengan train_df (jika ada kategori Title yang hilang di test)
test_df = test_df.reindex(
    columns=[col for col in train_df.columns if col != "Survived"], fill_value=0
)

In [17]:
# 2. Definisikan Fitur Baru
fitur_baru = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "FamilySize",
    "IsAlone",
    "Embarked_Q",
    "Embarked_S",
    "Title_Miss",
    "Title_Mr",
    "Title_Mrs",
    "Title_Rare",
]

X = train_df[fitur_baru]
y = train_df["Survived"]

# 3. Validasi Ulang
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=100, max_depth=5, min_samples_split=5, random_state=42
)
model.fit(X_train, y_train)

val_predictions = model.predict(X_val)
acc = accuracy_score(y_val, val_predictions)

print(f"Validation Accuracy Setelah Feature Engineering: {acc * 100:.2f}%")

Validation Accuracy Setelah Feature Engineering: 81.56%


In [18]:
import pandas as pd

# Ambil bobot kepentingan fitur
importance = pd.Series(model.feature_importances_, index=fitur_baru).sort_values(
    ascending=False
)
print("Tingkat Kepentingan Fitur:")
print(importance)

Tingkat Kepentingan Fitur:
Sex           0.265195
Title_Mr      0.209151
Pclass        0.116764
Fare          0.107269
Age           0.076955
FamilySize    0.065101
Title_Mrs     0.063687
Title_Miss    0.046951
Embarked_S    0.017928
IsAlone       0.013106
Embarked_Q    0.009654
Title_Rare    0.008239
dtype: float64


In [19]:
# 1. Pastikan fitur pada data test sama persis urutannya dengan data latih
X_test = test_df[fitur_baru]

# 2. Latih model dengan seluruh data train (100% data train) agar hasil optimal
model.fit(X, y)

# 3. Prediksi keselamatan untuk penumpang di test.csv
predictions = model.predict(X_test)

# 4. Buat DataFrame submission sesuai format resmi
submission = pd.DataFrame(
    {"PassengerId": test_df["PassengerId"], "Survived": predictions}
)

# 5. Simpan ke file CSV
submission.to_csv("submission.csv", index=False)

# 6. Tinjau 5 baris pertama dan total baris
print("5 Baris Pertama Submission:")
print(submission.head())
print(f"\nTotal baris submission: {len(submission)}")

5 Baris Pertama Submission:
   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1

Total baris submission: 418
